# UK Biobank blood traits with fewer credible sets from SuSiE-RSS

In the sketch LD fits from `18` (`UKB_fine.R`, `UKB_coloclization`), SuSiE-RSS reported fewer credible sets than SuSiE-rRSS for the region and trait pairs below. Refit each pair from the saved inputs.

| Region (GRCh38) | Trait |
|---|---|
| chr11:122,173,839–123,183,181 | Plateletcrit |
| chr11:15,756,615–17,320,581 | Reticulocyte % |
| chr5:175,313,842–176,069,996 | RDW |
| chr9:86,078,928–86,867,265 | PDW |
| chr13:49,803,063–51,789,698 | Lymphocyte % |
| chr22:27,931,399–31,198,752 | WBC count |

Run these steps in order in the same job.

1. `[ukb_case_inputs]` (one task per region × trait): rebuilds that trait's inputs exactly as `UKB_fine.R` did. The sumstat file is the one recorded in `res_<region>.rds` (`original_file_path`); `18` named that file by the region in the sumstat file name but loaded the sketch LD for the span of the variant positions (`region` stored in the result), so the same is done here. Steps: dbSNP positions; sketch LD from `ld_meta_file`, scaled; allele QC against the sketch panel (`sketch_harmonize`); kriging fix of the z-scores (`kriging_fix`, \|z\| > 2 and logLR > 2 flipped); in-sample z and LD on the harmonised variants. Saves `<unit>.inputs.rds` in the format of the `rss_qc/cases/eb_wrong` files (`insample_z`, `insample_LD`, `z_sketch`, `sketch`, `sketch_LD`, `n`, `B_sketch`), plus the harmonised sumstats and bookkeeping.
2. `[ukb_case_finemap]` (one task per region × trait × method), reading the inputs in a fresh session:
   - `susie_rss`: SuSiE-RSS on the sketch (`X = sketch`);
   - `susie_rrss`: SuSiE-rRSS on the sketch (`R_finite = B_sketch` = 10,000, `R_mismatch = "eb_mix"`), saving the pipeline result (`fit`: SER model when `R_reliability_flag` is set), the fit before fallback (`full_fit`) and the flag;
   - `insample`: SuSiE-RSS with the in-sample LD, the reference fit for comparing credible sets.

   Same calls as `UKB_fine.R` (`insample_res`, `susie_rss_res`, `susie_eb_mix_res`). Each output also records the run time of the fit.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_power_cases/result/<unit>/` (`<unit>.inputs.rds`, `<unit>.<method>.rds`); `<unit>` = `<chr>_<start>_<end>.<trait>`.

## 1. Inputs

In [ ]:
[ukb_case_inputs]
# One task per region x trait: rebuild the inputs of the UKB_fine.R (18) fits for this trait and save them;
# the fit step reads this file in a fresh session.
parameter: units = ["chr11_122173839_123183181.Plateletcrit",
                    "chr11_15756615_17320581.Reticulocyte_perc",
                    "chr5_175313842_176069996.RDW",
                    "chr9_86078928_86867265.PDW",
                    "chr13_49803063_51789698.Lymphocyte_perc",
                    "chr22_27931399_31198752.WBC_count"]
# <chr>_<start>_<end>.<trait>: the region as named by 18 (res_chr<chr>:<start>-<end>.rds) and the UKB blood trait
parameter: res_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/result")
parameter: snp_info_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/snp_info")
parameter: ld_meta_file = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: B_sketch = 10000
# R_finite for SuSiE-rRSS, as in UKB_fine.R
parameter: ld_ref = "ADSP"
# LD reference: "ADSP" (sketch, as 18 UKB_coloclization) or "1000G" (1000 Genomes EUR genotypes, as 19
# UKB_coloclization_1000_genome; pass --res_dir .../UKB_coloclization_1000_genome/result)
parameter: kg_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_EUR_hg38")
parameter: B_kg = 489
# R_finite for SuSiE-rRSS with the 1000G LD, as in 19
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "10h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: for_each = "units"
output: f'{cwd:a}/{_units}/{_units}.inputs.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_units}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(matrixStats)
    library(mixsqp)
    library(BEDMatrix)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    # susieR GitHub version (kriging_rss)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    ## ---- helpers, as in UKB_fine.R (18) ----
    kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
      krig <- kriging_rss(z = z, R = LD, n = n)
      krig_outlier <- krig$conditional_dist %>%
        mutate(index = row_number()) %>%
        filter(abs(z) > z_thres & logLR > logLR_thres)
      z_fix <- z
      z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
      list(z = z_fix, outlier_index = krig_outlier$index, outlier_number = length(krig_outlier$index))
    }

    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx  <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2, flipped_prop = length(flip_index) / ncol(sketch2))
    }

    ## ---- unit -> region, trait ----
    unit   <- "${_units}"
    block  <- str_split_fixed(unit, "\\.", 2)[, 1]
    trait  <- str_split_fixed(unit, "\\.", 2)[, 2]
    blk    <- str_split_fixed(block, "_", 3)
    region <- sprintf("%s:%s-%s", blk[, 1], blk[, 2], blk[, 3])
    chrom  <- str_remove(blk[, 1], "chr")

    ## ---- the sumstat file that 18 used for this region (recorded in its result) ----
    res_file <- file.path(${res_dir:r}, paste0("res_", region, ".rds"))
    if (!file.exists(res_file))
      stop(sprintf("no res_%s.rds; results on chr%s: %s", region, chrom,
                   paste(list.files(${res_dir:r}, pattern = paste0("^res_chr", chrom, ":")), collapse = ", ")))
    res <- readRDS(res_file)
    if (!trait %in% names(res)) stop(sprintf("%s not in res_%s.rds (traits: %s)", trait, region, paste(names(res), collapse = ", ")))
    file      <- res[[trait]]$original_file_path
    ld_region <- res[[trait]]$region             # 18 named the file by the sumstat file's region, but loaded LD for this one
    rm(res); invisible(gc())
    sumstat_list   <- readRDS(file)
    suff_stat_list <- readRDS(file.path(dirname(file), str_replace(basename(file), "summary", "sufficient")))
    tissue_idx     <- match(trait, colnames(sumstat_list$Z))
    n              <- suff_stat_list$N

    ## ---- variant positions (dbSNP, GRCh38); the LD region must match the one 18 used ----
    dpsnp_match  <- read_tsv(file.path(${snp_info_dir:r}, paste0("chrom_", chrom, "_snp.tsv")), show_col_types = FALSE)
    target_input <- sumstat_list$meta %>%
      left_join(dpsnp_match %>% select(SNP, BP), by = c("ID" = "SNP")) %>%
      select(CHR, BP, ID, REF, ALT, maf) %>%
      mutate(original_ID = paste0(ID, "_", REF)) %>%
      arrange(BP) %>%
      na.omit()
    ## 18 (ADSP) stored the span of the variant positions as region; 19 (1000G) stored the file's region
    ## but took the 1000G variants in that span
    ld_ref  <- "${ld_ref}"
    stopifnot(ld_ref %in% c("ADSP", "1000G"))
    ld_span <- sprintf("chr%s:%d-%d", chrom, min(target_input$BP), max(target_input$BP))
    if (ld_ref == "ADSP") stopifnot(ld_span == ld_region) else ld_region <- ld_span

    snp_meta <- target_input %>%
      left_join(tibble(z = sumstat_list$Z[, tissue_idx], original_ID = rownames(sumstat_list$Z)), by = "original_ID")
    colnames(snp_meta) <- c("chrom", "pos", "snp", "A2", "A1", "maf", "original_ID", "z")

    if (ld_ref == "ADSP") {
      ## ---- ADSP sketch LD (scaled), allele QC ----
      res_sketch           <- load_LD_matrix(${ld_meta_file:r}, ld_region, return_genotype = TRUE)
      res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
      harm  <- sketch_harmonize(target_data  = snp_meta,
                                ref_variants = res_sketch$ref_panel %>% select(-variant_id),
                                sketch       = res_sketch$LD_matrix)
      B_ref <- ${B_sketch}
    } else {
      ## ---- 1000G EUR genotypes in the span (missing rate 0.1, MAF 0.05), scaled, allele QC; as 19 ----
      snp_meta  <- na.omit(snp_meta)                     # 19 dropped variants with a missing z
      kg_prefix <- file.path(${kg_dir:r}, paste0("1000G_b38_EUR_chr", chrom))
      kg_geno   <- BEDMatrix(kg_prefix)
      kg_bim    <- read.delim(paste0(kg_prefix, ".bim"), header = FALSE, stringsAsFactors = FALSE)
      kg_gene   <- kg_bim %>% mutate(idx = row_number()) %>%
        filter(V4 >= min(target_input$BP) & V4 <= max(target_input$BP))
      kg_ids    <- paste0("chr", kg_gene$V1, ":", kg_gene$V4, ":", kg_gene$V6, ":", kg_gene$V5)
      kg_mat    <- data.matrix(kg_geno[, kg_gene$idx, drop = FALSE])
      colnames(kg_mat) <- kg_ids
      kg_mat    <- scale(filter_X(kg_mat, missing_rate_thresh = 0.1, maf_thresh = 0.05), center = TRUE, scale = TRUE)
      m         <- match(colnames(kg_mat), kg_ids)
      kg_ref    <- tibble(chrom = kg_gene$V1[m], pos = kg_gene$V4[m], A2 = kg_gene$V6[m], A1 = kg_gene$V5[m])
      harm  <- sketch_harmonize(target_data = snp_meta, ref_variants = kg_ref, sketch = kg_mat)
      B_ref <- ${B_kg}
    }
    if (ncol(harm$sketch) == 0) stop("no variants survive harmonisation")

    ## ---- reference LD, kriging fix ----
    sketch_LD      <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)
    krig           <- kriging_fix(z = harm$sumstat$z, LD = sketch_LD, n = n)
    harm$sumstat$z <- krig$z

    ## ---- in-sample z / LD on the harmonised variants ----
    insample_z  <- sumstat_list$Z[harm$sumstat$original_ID, tissue_idx]
    insample_LD <- sumstat_list$LD[harm$sumstat$original_ID, harm$sumstat$original_ID]

    inputs <- list(
      unit = unit, region = region, ld_region = ld_region, trait = trait, file = file,
      ld_ref = ld_ref, ld_input = if (ld_ref == "ADSP") "X" else "R",   # fit with X = sketch (18) or R = sketch_LD (19)
      sumstat     = as_tibble(harm$sumstat),     # harmonised, z after the kriging fix
      variant_ids = colnames(harm$sketch),
      z_sketch    = harm$sumstat$z,  sketch = harm$sketch, sketch_LD = sketch_LD,
      insample_z  = insample_z,      insample_LD = insample_LD,
      n = n, B_sketch = B_ref, sketch_rows = nrow(harm$sketch),
      flipped_prop = harm$flipped_prop,
      kriging_outlier_index = krig$outlier_index)
    saveRDS(inputs, ${_output:r})
    cat(sprintf("%s (%s LD): p = %d, n = %.0f, sketch rows = %d, flipped = %.3f, kriging outliers = %d, max |z| = %.2f\n",
                unit, ld_ref, ncol(harm$sketch), n, nrow(harm$sketch), harm$flipped_prop, krig$outlier_number, max(abs(inputs$z_sketch))))

## 2. Fine-mapping

In [ ]:
[ukb_case_finemap]
# One task per (region x trait, method), in a fresh R session that reads [ukb_case_inputs]; same calls as UKB_fine.R (18).
parameter: units = ["chr11_122173839_123183181.Plateletcrit",
                    "chr11_15756615_17320581.Reticulocyte_perc",
                    "chr5_175313842_176069996.RDW",
                    "chr9_86078928_86867265.PDW",
                    "chr13_49803063_51789698.Lymphocyte_perc",
                    "chr22_27931399_31198752.WBC_count"]
# <chr>_<start>_<end>.<trait>: the region as named by 18 (res_chr<chr>:<start>-<end>.rds) and the UKB blood trait
parameter: methods = ["susie_rss", "susie_rrss", "insample"]
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "20h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: for_each = ["units", "methods"]
output: f'{cwd:a}/{_units}/{_units}.{_methods}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_units}_{_methods}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(MASS)
    library(crayon)                 # used by the sourced susieR (combine_styles)
    library(Rfast)

    # susieR GitHub version (SuSiE-RSS / SuSiE-rRSS with eb_mix)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    method <- "${_methods}"
    inp    <- readRDS(file.path(${_output:dr}, "${_units}.inputs.rds"))
    cat("unit:", inp$unit, " method:", method, " p =", length(inp$z_sketch), "\n")

    ## reference LD as the original fit used it: X = sketch (18, ADSP) or R = sketch_LD (19, 1000G)
    ld_arg <- if (identical(inp$ld_input, "R")) list(R = inp$sketch_LD) else list(X = inp$sketch)
    rss_ref <- function(...) do.call(susie_rss, c(list(z = inp$z_sketch, n = inp$n), ld_arg, list(...)))

    t0  <- Sys.time()
    out <- tryCatch(switch(method,
      susie_rss = list(fit = rss_ref()),
      susie_rrss = {
        fit  <- rss_ref(R_finite = inp$B_sketch, R_mismatch = "eb_mix")
        flag <- isTRUE(fit$R_finite_diagnostics$R_reliability_flag)
        if (flag && is.null(fit$R_finite_diagnostics$ser_model)) stop("R_reliability_flag set but no ser_model in the fit")
        list(fit = if (flag) fit$R_finite_diagnostics$ser_model else fit,    # pipeline result
             full_fit = fit, ser_fallback = flag)                            # fit before fallback, flag
      },
      insample = list(fit = susie_rss(z = inp$insample_z, R = inp$insample_LD, n = inp$n)),
      stop("unknown method: ", method)),
      error = function(e) list(error = conditionMessage(e)))
    out$time_s <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    out$unit <- inp$unit; out$region <- inp$region; out$trait <- inp$trait; out$method <- method
    out$variant_ids <- inp$variant_ids
    saveRDS(out, ${_output:r})
    if (!is.null(out$error)) cat("[FAIL]", out$error, "\n") else
      cat(sprintf("%s %s: %d CS, %.1f s%s\n", inp$unit, method, length(out$fit$sets$cs), out$time_s,
                  if (isTRUE(out$ser_fallback)) " (SER fallback)" else ""))

Submit: inputs, then the three fits, in one job.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKB_power_cases"
job_dir="${base_dir}/${job_name}"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/31.UKB_Power_Cases.ipynb"
sos run "$nb" ukb_case_inputs  --cwd WORK_DIR/result --mem 60G -J 3 && \
sos run "$nb" ukb_case_finemap --cwd WORK_DIR/result --mem 60G -J 3
EOF

sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" ${job_name}.sh > ${job_name}.sh.submit
sb ${job_name}.sh.submit

## 3. Check

Credible sets per region × trait and method (pipeline result for SuSiE-rRSS, with the count before fallback), SER fallback, and run time.

In [1]:
library(tidyverse)
`%||%` <- function(a, b) if (is.null(a)) b else a
root <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_power_cases/result"

fits <- list.files(root, pattern = "\\.(susie_rss|susie_rrss|insample)\\.rds$", recursive = TRUE, full.names = TRUE)
map_dfr(fits, function(f) {
  o <- readRDS(f)
  tibble(unit = o$unit, method = o$method, error = o$error %||% NA_character_,
         n_cs = length(o$fit$sets$cs), n_cs_before_fallback = if (!is.null(o$full_fit)) length(o$full_fit$sets$cs) else NA_integer_,
         ser_fallback = o$ser_fallback %||% NA, time_s = round(o$time_s, 1))
}) %>%
  arrange(unit, factor(method, levels = c("insample", "susie_rss", "susie_rrss"))) %>%
  as.data.frame()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


unit,method,error,n_cs,n_cs_before_fallback,ser_fallback,time_s
<chr>,<chr>,<chr>,<int>,<int>,<lgl>,<dbl>
chr11_122173839_123183181.Plateletcrit,insample,NA,2,NA,NA,50.6
chr11_122173839_123183181.Plateletcrit,susie_rss,NA,1,NA,NA,45.6
chr11_122173839_123183181.Plateletcrit,susie_rrss,NA,2,2,FALSE,244.6
chr11_15756615_17320581.Reticulocyte_perc,insample,NA,3,NA,NA,57.6
chr11_15756615_17320581.Reticulocyte_perc,susie_rss,NA,3,NA,NA,49.0
chr11_15756615_17320581.Reticulocyte_perc,susie_rrss,NA,2,2,FALSE,211.1
chr13_49803063_51789698.Lymphocyte_perc,insample,NA,2,NA,NA,68.1
chr13_49803063_51789698.Lymphocyte_perc,susie_rss,NA,1,NA,NA,70.6
chr13_49803063_51789698.Lymphocyte_perc,susie_rrss,NA,2,2,FALSE,370.2
